# Day 16 — Type casting in Polars: strict vs non-strict

*100 Days of Polars · Phase 2: Expressions*

- **Post:** {{TODO: Medium link}}
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [33]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


`cast` changes a column's dtype. By default it is **strict**: one value that cannot be converted fails the whole cast.
`strict=False` turns those values into nulls instead. Which one you want depends on whether bad data should stop the pipeline.

In [34]:
items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
customers = pl.read_csv("data/raw/olist_customers_dataset.csv")
reviews = pl.read_csv("data/raw/olist_order_reviews_dataset.csv")  # no try_parse_dates on purpose

## 1. Numeric casts

In [35]:
items.select(
    "price",
    as_f32=pl.col("price").cast(pl.Float32),
    as_int=pl.col("price").cast(pl.Int32),  # float -> int truncates toward zero
).head(5)

price,as_f32,as_int
f64,f32,i32
58.9,58.900002,58
239.9,239.899994,239
199.0,199.0,199
12.99,12.99,12
199.9,199.899994,199


In [36]:
# Booleans cast to integers (handy for counting)
items.select((pl.col("price") > 100).cast(pl.UInt32).sum())

price
u32
40313


## 2. Strict vs non-strict

In [37]:
raw = pl.DataFrame({"qty": ["10", "20", "3x", None, "7"]})

try:
    raw.select(pl.col("qty").cast(pl.Int64))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

InvalidOperationError -> conversion from `str` to `i64` failed in column 'qty' for 1 out of 5 values: ["3x"]


In [38]:
raw.select(pl.col("qty").cast(pl.Int64, strict=False))

qty
i64
10
20
null
null
7


Non-strict casting hides the bad value among real nulls. In a pipeline, **find the rows that failed** before you move on:

In [39]:
raw.filter(
    pl.col("qty").is_not_null(),                          # had a value...
    pl.col("qty").cast(pl.Int64, strict=False).is_null(),  # ...that did not survive the cast
)

qty
str
"""3x"""


## 3. Overflow

In [40]:
s = pl.Series("n", [100, 255, 300])

try:
    s.cast(pl.UInt8)
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

print(s.cast(pl.UInt8, strict=False).to_list())        # out of range -> null
print(s.cast(pl.UInt8, wrap_numerical=True).to_list())  # wrap around, like C

InvalidOperationError -> conversion from `i64` to `u8` failed in column 'n' for 1 out of 3 values: [300]
[100, 255, None]
[100, 255, 44]


## 4. Shrinking types to save memory

In [41]:
scores = reviews.select("review_score")
small = scores.with_columns(pl.col("review_score").cast(pl.UInt8))

print(scores.schema, round(scores.estimated_size("kb"), 1), "KB")
print(small.schema, round(small.estimated_size("kb"), 1), "KB")

Schema({'review_score': Int64}) 775.2 KB
Schema({'review_score': UInt8}) 96.9 KB


## 5. Numbers that are really strings: zip codes

In [42]:
print(customers.schema["customer_zip_code_prefix"])
customers.select(
    "customer_zip_code_prefix",
    zip_str=pl.col("customer_zip_code_prefix").cast(pl.String).str.zfill(5),  # put back the leading zero
).sort("customer_zip_code_prefix").head(3)

Int64


customer_zip_code_prefix,zip_str
i64,str
1003,"""01003"""
1004,"""01004"""
1004,"""01004"""


## 6. Strings to dates: use `str.to_*`, not `cast`

Polars 2.0 removed casting from `String` to `Date`, `Datetime` and `Time`. Parse with `str.to_date`, `str.to_datetime`, `str.to_time` instead; they take a `format` and their own `strict` flag.

In [43]:
print(reviews.schema["review_creation_date"])

try:
    reviews.select(pl.col("review_creation_date").cast(pl.Date))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

String
InvalidOperationError -> casting from string to date is not supported.


In [44]:
reviews.select(
    created=pl.col("review_creation_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    answered=pl.col("review_answer_timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S"),
).head(3)

created,answered
datetime[μs],datetime[μs]
2018-01-18 00:00:00,2018-01-18 21:46:59
2018-03-10 00:00:00,2018-03-11 03:05:13
2018-02-17 00:00:00,2018-02-18 14:36:24


**A date from a datetime string.** `str.to_date("%Y-%m-%d")` on `"2018-01-18 21:46:59"` fails: by default the format must match the **whole** string (`exact=True`). Four ways that work:

In [45]:
ts = pl.col("review_answer_timestamp")
try:
    reviews.select(ts.str.to_date("%Y-%m-%d"))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

reviews.select(
    a=ts.str.to_datetime("%Y-%m-%d %H:%M:%S").dt.date(),   # 1. parse fully, take the date part
    b=ts.str.to_date("%Y-%m-%d %H:%M:%S"),                 # 2. full format, time dropped
    c=ts.str.slice(0, 10).str.to_date("%Y-%m-%d"),         # 3. cut to 10 characters first
    d=ts.str.to_date("%Y-%m-%d", exact=False),             # 4. format may match part of the string (slower)
).head(3)

InvalidOperationError -> conversion from `str` to `date` failed in column 'review_answer_timestamp' for 447 out of 447 values: ["2018-01-18 21:46:59", "2018-03-11 03:05:13", … "2018-05-23 16:45:47"]


a,b,c,d
date,date,date,date
2018-01-18,2018-01-18,2018-01-18,2018-01-18
2018-03-11,2018-03-11,2018-03-11,2018-03-11
2018-02-18,2018-02-18,2018-02-18,2018-02-18


**`exact` vs `strict`: two steps on every row.**

1. `exact` decides whether the row parses: `True` = the format must match the **complete** string, `False` = **part** of it is enough.
2. `strict` decides what happens to a row that did not parse: `True` = raise, `False` = null.

`exact` never makes nulls by itself. The 2×2 grid on Olist (the `exact=True, strict=True` cell raises, so it runs separately):

In [46]:
reviews.select(
    raw=ts,
    exact_F_strict_T=ts.str.to_date("%Y-%m-%d", exact=False, strict=True),
    exact_F_strict_F=ts.str.to_date("%Y-%m-%d", exact=False, strict=False),
    exact_T_strict_F=ts.str.to_date("%Y-%m-%d", exact=True,  strict=False),   # the trap: every row null
).head(5)

raw,exact_F_strict_T,exact_F_strict_F,exact_T_strict_F
str,date,date,date
"""2018-01-18 21:46:59""",2018-01-18,2018-01-18,null
"""2018-03-11 03:05:13""",2018-03-11,2018-03-11,null
"""2018-02-18 14:36:24""",2018-02-18,2018-02-18,null
"""2017-04-21 22:02:06""",2017-04-21,2017-04-21,null
"""2018-03-02 10:26:53""",2018-03-02,2018-03-02,null


In [47]:
try:
    reviews.select(ts.str.to_date("%Y-%m-%d", exact=True, strict=True))
except Exception as err:
    print("exact=True, strict=True ->", type(err).__name__)

exact=True, strict=True -> InvalidOperationError


In [48]:
reviews.select(
    source_nulls=ts.null_count(),
    parsed_nulls=ts.str.to_date("%Y-%m-%d", exact=False, strict=False).null_count(),
    strict_false_only=ts.str.to_date("%Y-%m-%d", strict=False).null_count(),   # the trap: exact match never happens
    rows=pl.len(),
)

source_nulls,parsed_nulls,strict_false_only,rows
u32,u32,u32,u32
0,0,99224,99224


In [49]:
# With a genuinely bad row, the difference shows
bad = pl.DataFrame({"ts": ["2018-01-18 21:46:59", "2018-03-11 03:05:13", "not a date"]})
try:
    bad.select(pl.col("ts").str.to_date("%Y-%m-%d", exact=False))              # raises
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])
bad.select(pl.col("ts").str.to_date("%Y-%m-%d", exact=False, strict=False))   # 2018-01-18, 2018-03-11, null

InvalidOperationError -> conversion from `str` to `date` failed in column 'ts' for 1 out of 3 values: ["not a date"]


ts
date
2018-01-18
2018-03-11
null


## Try it yourself

1. Cast `order_item_id` in `items` to the smallest unsigned integer that fits. How much memory do you save?
2. Build a column of prices as strings with a few `"N/A"` values and report the failed rows, as in section 2.
3. Parse `review_creation_date` with `str.to_date` and a format. What happens to the time part?